# Solution 3.1: Aggregating and Summarizing (Angola trade)

Angola's international trade of goods, published by INE. Twenty two years of
exports and imports, partner by partner, in thousands of US dollars.

You will practice: `groupby`, `.agg`, grouping by two variables, `stack` and
`unstack`, pivot tables, cross tabulations, and filtering groups.

**PT:** O comercio internacional de bens de Angola, publicado pelo INE. Vinte e
dois anos de exportacoes e importacoes, parceiro a parceiro, em milhares de
dolares.

Vai praticar: `groupby`, `.agg`, agrupar por duas variaveis, `stack` e `unstack`,
tabelas dinamicas, tabelas de contingencia, e filtrar grupos.

> **Pipeline:** reads `0_raw/angola/international_trade`, writes `20_processed/`.

### Path Setup (run first)

**PT:** Configuracao dos caminhos.

In [1]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_RAW_DIR = '../../data/0_raw/angola'
DATA_PROC_DIR = '../../data/20_processed'

TRADE_DIR = 'international_trade'
PARTNERS_FILE = 'Comercio Externo de Bens por Países Parceiros.xlsx'

trade_dir = os.path.join(DATA_RAW_DIR, TRADE_DIR)
partners_path = os.path.join(trade_dir, PARTNERS_FILE)

pd.set_option('display.float_format', lambda x: f'{x:,.1f}')
print('File:', partners_path)
print('Exists?:', os.path.exists(partners_path))

File: ../../data/0_raw/angola/international_trade/Comercio Externo de Bens por Países Parceiros.xlsx
Exists?: True


---

## Task 1: Load both flows into one table

The workbook keeps exports and imports on separate sheets with the same shape.
Load both, tidy the column names, and stack them into one table with a `flow`
column saying which is which.

**What to do:** load each sheet with `skiprows=2`, convert the names to snake
case, keep only the rows where the country name is filled in, add `flow`, then
concatenate.

**PT:** O ficheiro tem exportacoes e importacoes em folhas separadas com a mesma
forma. Carregue as duas, arrume os nomes, e empilhe numa so tabela com uma coluna
`flow`.

**O que fazer:** carregue cada folha com `skiprows=2`, converta os nomes para
snake case, mantenha as linhas com o nome do pais preenchido, adicione `flow`, e
concatene.

In [2]:
def to_snake_case(columns):
    """Lower case, strip accents, join words with underscores.

    Minusculas, sem acentos, palavras unidas por underscore.
    """
    return (columns
            .str.replace('\n', ' ', regex=False)
            .str.strip()
            .str.lower()
            .str.normalize('NFKD')
            .str.encode('ascii', errors='ignore')
            .str.decode('utf-8')
            .str.replace(' ', '_', regex=False))


sheets = {'Exportação por Países (USD)': 'Export',
          'Importação por Países (USD)': 'Import'}

frames = []
for sheet_name, flow in sheets.items():
    sheet = pd.read_excel(partners_path, sheet_name=sheet_name, skiprows=2)
    sheet.columns = to_snake_case(sheet.columns)
    sheet = sheet[sheet['pais'].notna()].copy()
    sheet['flow'] = flow
    frames.append(sheet)

wide = pd.concat(frames, ignore_index=True)
print('wide:', wide.shape)
print(wide['flow'].value_counts())

wide: (498, 25)
flow
Export    249
Import    249
Name: count, dtype: int64


**Answers:**

- 498 rows, 249 per flow: 248 partner countries plus `ZZ`, Desconhecido.
- The year columns are still columns, one per year from `ano_2004` to `ano_2025`.
  That shape is fine for reading but useless for grouping, which Task 2 fixes.
- Adding `flow` before the concat is what keeps the two halves distinguishable.

**PT:** 498 linhas, 249 por fluxo: 248 paises mais `ZZ`, Desconhecido. As colunas
de ano continuam colunas, de `ano_2004` a `ano_2025`, o que serve para ler mas
nao para agrupar. Adicionar `flow` antes do concat e o que mantem as duas metades
distinguiveis.

---

## Task 2: Turn the year columns into rows with `stack()`

`groupby` needs the thing you group by to be a column, not a column *name*. Right
now the year lives in the header, so there is nothing to group.

`stack()` pushes columns down into the index, making the table longer and
narrower. It is the exact opposite of `unstack()`, which you will use in Task 5.

**What to do:** set the identifying columns as the index, keep only the year
columns, `stack()` them, reset the index, name the columns, and turn the year
text into a number.

**PT:** O `groupby` precisa que aquilo que agrupa seja uma coluna, e nao o nome
de uma coluna. Agora o ano esta no cabecalho, por isso nao ha nada para agrupar.

`stack()` empurra as colunas para o indice, tornando a tabela mais comprida e
estreita. E o oposto de `unstack()`, que vai usar na Tarefa 5.

**O que fazer:** ponha as colunas de identificacao no indice, fique so com as
colunas de ano, faca `stack()`, reponha o indice, dê nome as colunas, e converta
o texto do ano em numero.

In [3]:
year_columns = [col for col in wide.columns if col.startswith('ano_')]
print('Year columns:', len(year_columns), year_columns[:3], '...', year_columns[-1:])

trade = (wide
         .set_index(['codigo', 'pais', 'flow'])[year_columns]
         .stack()
         .reset_index())
trade.columns = ['country_code', 'country_name', 'flow', 'year', 'value_thousand_usd']

# The year arrives as the text 'ano_2004' / O ano chega como o texto 'ano_2004'
trade['year'] = trade['year'].str.replace('ano_', '', regex=False).astype(int)

print('long:', trade.shape)
trade.head()

Year columns: 22 ['ano_2004', 'ano_2005', 'ano_2006'] ... ['ano_2025']
long: (10956, 5)


,country_code,country_name,flow,year,value_thousand_usd
0,AF,Afeganistão,Export,2004,15.0
1,AF,Afeganistão,Export,2005,0.0
2,AF,Afeganistão,Export,2006,0.0
3,AF,Afeganistão,Export,2007,0.0
4,AF,Afeganistão,Export,2008,0.0


**Answers:**

- 10,956 rows: 249 partners times 22 years times 2 flows.
- Each row is now one partner, one year, one flow, which is exactly what
  `groupby` needs.
- `stack()` lengthens, `unstack()` widens. Wide tables are easier for people to
  read; long tables are easier for machines to group, plot and store.

**PT:** 10.956 linhas: 249 parceiros vezes 22 anos vezes 2 fluxos. Cada linha e
agora um parceiro, um ano, um fluxo, que e o que o `groupby` precisa. `stack()`
alonga, `unstack()` alarga.

---

## Task 3: First groupings

`groupby` splits the table into groups, applies a function to each, and puts the
results back together.

**What to do:** total the value by flow, then by year, and count the rows per
flow with `size()`.

**PT:** O `groupby` divide a tabela em grupos, aplica uma funcao a cada um, e
junta os resultados.

**O que fazer:** some o valor por fluxo, depois por ano, e conte as linhas por
fluxo com `size()`.

In [4]:
print('Total by flow / Total por fluxo (thousand USD):')
print(trade.groupby('flow')['value_thousand_usd'].sum())

Total by flow / Total por fluxo (thousand USD):
flow
Export   918,770,854.1
Import   385,425,558.4
Name: value_thousand_usd, dtype: float64


In [5]:
print('Rows per flow / Linhas por fluxo:')
print(trade.groupby('flow').size())

Rows per flow / Linhas por fluxo:
flow
Export    5478
Import    5478
dtype: int64


In [6]:
by_year = trade.groupby('year')['value_thousand_usd'].sum()
print(by_year.tail(6))

year
2020   30,262,549.3
2021   45,191,204.1
2022   67,597,242.0
2023   52,335,764.6
2024   51,425,789.1
2025   47,486,268.0
Name: value_thousand_usd, dtype: float64


**Answers:**

- Exports total 918,770,854 thousand USD over the 22 years, imports 385,425,558.
  Angola sells abroad far more than it buys, which is what an oil exporter looks
  like.
- `size()` returns 5,478 rows per flow, and it counts rows rather than summing a
  column, so it answers "how many records" and not "how much".
- Trade by year is not a smooth trend: it rises to a peak and falls back, and
  2025 is the lowest of the recent years at 47,486,268.

**PT:** As exportacoes somam 918.770.854 milhares de dolares em 22 anos e as
importacoes 385.425.558. `size()` devolve 5.478 linhas por fluxo e conta linhas,
nao soma valores. O comercio por ano nao e uma tendencia suave.

---

## Task 4: Several statistics at once with `.agg()`

One number per group rarely tells the whole story. `.agg()` gives you several in
one pass, and named aggregations make the output readable.

**What to do:** for exports only, compute the mean, median, standard deviation
and count by year, then repeat with named aggregations.

**PT:** Um so numero por grupo raramente conta a historia toda. `.agg()` da
varios de uma vez, e nomear as agregacoes torna o resultado legivel.

**O que fazer:** so para as exportacoes, calcule media, mediana, desvio padrao e
contagem por ano, e depois repita com agregacoes nomeadas.

In [7]:
exports = trade[trade['flow'] == 'Export']

exports.groupby('year')['value_thousand_usd'].agg(['mean', 'median', 'std', 'count']).tail(5)

,mean,median,std,count
year,,,,
2021,"135,514.8",1.3,"1,308,015.1",249
2022,"201,222.3",0.5,"1,473,690.6",249
2023,"147,706.5",1.0,"1,178,162.1",249
2024,"146,387.8",0.9,"1,088,665.2",249
2025,"123,405.2",6.4,"966,916.5",249


In [8]:
exports.groupby('year')['value_thousand_usd'].agg(
    average='mean',
    middle='median',
    spread='std',
    partners='count',
).tail(5)

,average,middle,spread,partners
year,,,,
2021,"135,514.8",1.3,"1,308,015.1",249
2022,"201,222.3",0.5,"1,473,690.6",249
2023,"147,706.5",1.0,"1,178,162.1",249
2024,"146,387.8",0.9,"1,088,665.2",249
2025,"123,405.2",6.4,"966,916.5",249


**Answers:**

- In 2025 the mean export per partner is 123,405.2 thousand USD and the median is
  **6.4**. The mean is roughly twenty thousand times the median.
- That gap is the finding, not a mistake. Angola exports almost everything to a
  handful of partners, so most partner rows are near zero while a few are
  enormous. The standard deviation, close to a million, says the same thing.
- Reporting the mean alone here would be badly misleading. For a concentrated
  distribution the median describes the typical partner and the mean describes
  the total divided up.

**PT:** Em 2025 a media por parceiro e 123.405,2 milhares de dolares e a mediana e
**6,4**. A diferenca e o resultado, nao um erro: Angola exporta quase tudo para
poucos parceiros. Publicar so a media seria enganador.

---

## Task 5: Group by two variables, then `unstack()`

Grouping by two columns gives a result with two index levels, which reads poorly.
`unstack()` moves the innermost level up into the columns and turns it into a
table.

**What to do:** total the value by year and flow, look at the stacked result,
then `unstack()` it into a year by flow table.

**PT:** Agrupar por duas colunas da um resultado com dois niveis de indice, que
se le mal. `unstack()` move o nivel de dentro para as colunas.

**O que fazer:** some o valor por ano e fluxo, veja o resultado empilhado, e
depois faca `unstack()` para obter uma tabela de ano por fluxo.

In [9]:
by_year_flow = trade.groupby(['year', 'flow'])['value_thousand_usd'].sum()
print(by_year_flow.tail(6))

year  flow  
2023  Export   36,778,930.2
      Import   15,556,834.4
2024  Export   36,450,558.1
      Import   14,975,231.0
2025  Export   30,727,896.0
      Import   16,758,372.0
Name: value_thousand_usd, dtype: float64


In [10]:
table = by_year_flow.unstack()
table.tail(6)

flow,Export,Import
year,,
2020,"21,051,986.6","9,210,562.7"
2021,"33,743,174.1","11,448,029.9"
2022,"50,104,357.6","17,492,884.5"
2023,"36,778,930.2","15,556,834.4"
2024,"36,450,558.1","14,975,231.0"
2025,"30,727,896.0","16,758,372.0"


In [11]:
# A balance column falls out once the flows are side by side
# Com os fluxos lado a lado, a balanca comercial sai naturalmente
table['balance'] = table['Export'] - table['Import']
table.tail(6)

flow,Export,Import,balance
year,,,
2020,"21,051,986.6","9,210,562.7","11,841,424.0"
2021,"33,743,174.1","11,448,029.9","22,295,144.2"
2022,"50,104,357.6","17,492,884.5","32,611,473.1"
2023,"36,778,930.2","15,556,834.4","21,222,095.8"
2024,"36,450,558.1","14,975,231.0","21,475,327.1"
2025,"30,727,896.0","16,758,372.0","13,969,524.0"


**Answers:**

- The grouped result has one row per year and flow pair, 44 rows in total. After
  `unstack()` it is 22 rows by 2 columns, which is how you would publish it.
- In 2025 exports were 30,727,896 and imports 16,758,372, a surplus of
  13,969,524 thousand USD.
- The surplus has narrowed sharply: in 2022 exports were nearly three times
  imports, in 2025 less than twice.

**PT:** O resultado agrupado tem 44 linhas. Depois do `unstack()` sao 22 linhas
por 2 colunas, a forma como se publicaria. Em 2025 as exportacoes foram 30.727.896
e as importacoes 16.758.372, um excedente de 13.969.524. O excedente estreitou.

---

## Task 6: Pivot tables and margins

`pivot_table` does the grouping and the reshaping in one call, and `margins=True`
adds the totals row and column.

Careful: margins use the same `aggfunc`, so with `mean` they are averages and not
sums.

**What to do:** build a pivot table of the value by year and flow for the last
three years, with totals.

**PT:** O `pivot_table` faz o agrupamento e a remodelacao numa so chamada, e
`margins=True` acrescenta a linha e a coluna de totais, calculadas com a mesma
funcao.

**O que fazer:** construa uma tabela dinamica do valor por ano e fluxo para os
ultimos tres anos, com totais.

In [12]:
recent = trade[trade['year'] >= 2023]

pd.pivot_table(
    recent,
    values='value_thousand_usd',
    index='year',
    columns='flow',
    aggfunc='sum',
    margins=True,
    margins_name='Total',
)

flow,Export,Import,Total
year,,,
2023,"36,778,930.2","15,556,834.4","52,335,764.6"
2024,"36,450,558.1","14,975,231.0","51,425,789.1"
2025,"30,727,896.0","16,758,372.0","47,486,268.0"
Total,"103,957,384.3","47,290,437.4","151,247,821.7"


**Answers:**

- The bottom right cell, 151,247,822, is the total trade of the three years.
- The `Total` column is exports plus imports for that year, and the `Total` row is
  the three year total for that flow.
- With `aggfunc='mean'` the margins would be averages, and the `Total` row would
  not be the sum of the rows above it. That surprises people, so state which
  function produced a published table.
- `pivot_table` aggregates, `pivot` only reshapes and raises if a cell would hold
  more than one value.

**PT:** A celula inferior direita, 151.247.822, e o comercio total dos tres anos.
Com `aggfunc='mean'` as margens seriam medias e nao somas. `pivot_table` agrega,
`pivot` apenas remodela.

---

## Task 7: Cross tabulation of two categories

`crosstab` counts how often each combination of two categorical variables occurs.
Value is a number, not a category, so first turn it into a size band.

**What to do:** write `size_band` returning `'None'` for zero, `'Under 100'`,
`'100 to 10k'` and `'Over 10k'`, apply it, then cross tabulate `flow` against the
band, first as counts and then as row percentages.

**PT:** O `crosstab` conta com que frequencia ocorre cada combinacao de duas
variaveis categoricas. O valor e um numero, por isso primeiro converta-o numa
faixa.

**O que fazer:** escreva `size_band` que devolve `'None'` para zero, `'Under
100'`, `'100 to 10k'` e `'Over 10k'`, aplique, e faca a tabela cruzada de `flow`
contra a faixa, em contagens e depois em percentagens por linha.

In [13]:
def size_band(value):
    """Band one trade value / Classificar um valor de comercio."""
    if value <= 0:
        return 'None'
    if value < 100:
        return 'Under 100'
    if value < 10000:
        return '100 to 10k'
    return 'Over 10k'


trade['size_band'] = trade['value_thousand_usd'].apply(size_band)
pd.crosstab(trade['flow'], trade['size_band'])

size_band,100 to 10k,None,Over 10k,Under 100
flow,,,,
Export,821,2578,757,1322
Import,1877,1044,1218,1339


In [14]:
# Row percentages: each row sums to 100
# Percentagens por linha: cada linha soma 100
(pd.crosstab(trade['flow'], trade['size_band'], normalize='index') * 100).round(1)

size_band,100 to 10k,None,Over 10k,Under 100
flow,,,,
Export,15.0,47.1,13.8,24.1
Import,34.3,19.1,22.2,24.4


**Answers:**

- **47.1%** of export records are zero against **19.1%** of import records. In
  most years Angola sells nothing at all to most countries, while it buys
  something from far more of them.
- Read the other end too: 13.8% of export records are over 10k, against 22.2% of
  import records. Exports are fewer and larger, imports more numerous and smaller.
- `normalize='index'` gives row percentages, `'columns'` column percentages, and
  `'all'` percentages of the whole table. Always say which one a published table
  used, because the three tell different stories from the same counts.

**PT:** **47,1%** dos registos de exportacao sao zero contra **19,1%** dos de
importacao: Angola nao vende nada a maioria dos paises, mas compra a muitos mais.
`normalize='index'` da percentagens por linha, `'columns'` por coluna, e `'all'`
sobre o total.

---

## Task 8: Chain operations to answer a question

Aggregations chain. Group, then sort, then take the head, all in one expression.

**What to do:** find the ten partners with the largest total exports over the
whole period, then show mean, median and count per partner sorted by median.

**PT:** As agregacoes encadeiam-se: agrupar, ordenar, e ficar com as primeiras
linhas numa so expressao.

**O que fazer:** encontre os dez parceiros com maiores exportacoes totais no
periodo, e depois mostre media, mediana e contagem por parceiro ordenadas pela
mediana.

In [15]:
top_partners = (exports
                .groupby('country_name')['value_thousand_usd']
                .sum()
                .sort_values(ascending=False)
                .head(10))
top_partners

country_name
China                       406,979,040.3
Estados Unidos da América    99,377,832.4
Índia                        75,402,794.8
Canadá                       40,700,532.6
França                       34,491,337.2
Taiwan                       32,703,025.9
Espanha                      30,479,983.8
África do Sul                25,408,151.5
Países Baixos                23,604,338.1
Emirados Árabes Unidos       18,033,439.7
Name: value_thousand_usd, dtype: float64

In [16]:
(exports
 .groupby('country_name')['value_thousand_usd']
 .agg(['mean', 'median', 'count'])
 .sort_values('median', ascending=False)
 .head(10))

,mean,median,count
country_name,,,
China,"18,499,047.3","18,743,703.3",22
Índia,"3,427,399.8","3,255,320.1",22
Estados Unidos da América,"4,517,174.2","2,078,279.6",22
França,"1,567,788.1","1,446,010.3",22
África do Sul,"1,154,916.0","1,219,627.3",22
Taiwan,"1,486,501.2","1,145,363.5",22
Canadá,"1,850,024.2","1,130,286.4",22
Espanha,"1,385,453.8","1,114,175.8",22
Países Baixos,"1,072,924.5","959,901.4",22


**Answers:**

- China alone takes 406,979,040 thousand USD of exports over the period, more
  than four times the second partner, the United States, at 99,377,832.
- Sorting by total and sorting by median give different lists. The total finds the
  partners that matter to the economy; the median finds the partners Angola trades
  with consistently, year in and year out.
- Neither ranking is the right one on its own. Say which you used.

**PT:** So a China representa 406.979.040 milhares de dolares, mais de quatro
vezes o segundo parceiro. Ordenar pelo total e pela mediana da listas diferentes:
o total encontra quem pesa na economia, a mediana quem negoceia de forma
constante.

---

## Task 9: Filter on a property of the group

Sometimes the filter depends on the group and not on the row: keep only partners
that reach a certain size. That takes two steps, compute the group statistic,
then select the rows whose group qualifies.

**What to do:** total the trade per partner, keep the partners above 1,000,000
thousand USD, and filter the table down to them with `isin`.

**PT:** Por vezes o filtro depende do grupo e nao da linha. Sao dois passos:
calcular a estatistica do grupo e depois selecionar as linhas cujo grupo passa.

**O que fazer:** some o comercio por parceiro, fique com os que passam de
1.000.000 milhares de dolares, e filtre a tabela com `isin`.

In [17]:
per_partner = trade.groupby('country_name')['value_thousand_usd'].sum()
major = per_partner[per_partner > 1_000_000].index

print('Partners kept / Parceiros mantidos:', len(major), 'of', trade['country_name'].nunique())

trade_major = trade[trade['country_name'].isin(major)]
print('Rows:', len(trade), '->', len(trade_major))
print('Share of all trade kept: '
      f"{trade_major['value_thousand_usd'].sum() / trade['value_thousand_usd'].sum() * 100:.1f}%")

Partners kept / Parceiros mantidos: 45 of 249
Rows: 10956 -> 1980
Share of all trade kept: 98.2%


**Answers:**

- 45 partners of 249 clear the threshold, and they account for almost all the
  trade by value.
- That is the concentration the crosstab already hinted at, now measured: fewer
  than a fifth of the partners carry nearly all of the money.
- The threshold is a choice and it belongs in the footnote of any table built
  this way.

**PT:** 45 parceiros em 249 passam o limiar e representam quase todo o comercio
em valor. Menos de um quinto dos parceiros carrega quase todo o dinheiro. O
limiar e uma escolha e deve constar em nota de rodape.

---

## Task 10: Save

**What to do:** write the long table and the year by flow summary to
`20_processed/` with `index=False` for the long one.

**PT:** **O que fazer:** grave a tabela longa e o resumo por ano e fluxo em
`20_processed/`.

In [18]:
os.makedirs(DATA_PROC_DIR, exist_ok=True)

long_out = os.path.join(DATA_PROC_DIR, 'angola_trade_long.csv')
summary_out = os.path.join(DATA_PROC_DIR, 'angola_trade_year_flow.csv')

trade.to_csv(long_out, index=False)
table.to_csv(summary_out)

print('long:', trade.shape, '| summary:', table.shape)

long: (10956, 6) | summary: (22, 3)


**Answers:**

- The long table is the analysis ready form, one row per partner, year and flow.
  The summary is the published form, one row per year.
- The summary keeps its index because the year is the row label, so it is written
  without `index=False`.
- Nothing was written into `0_raw/`.

**PT:** A tabela longa e a forma pronta para analise, o resumo e a forma
publicada. O resumo mantem o indice porque o ano e a etiqueta da linha. Nada foi
escrito em `0_raw/`.